# RefMate — FASE 3: OCR con LightOnOCR-1B-1025

Convierte las imágenes recortadas de los PDFs a texto plano usando LightOnOCR.

## Flujo completo

**En tu máquina local (antes de este notebook):**
```bash
# FASE 2: generar imágenes recortadas
uv run python -m refmate.ingest.cropper

# Comprimir para subir a Colab
cd data && zip -r ../images.zip images/
```

**En este notebook:**
1. Runtime → Change runtime type → **T4 GPU**
2. Ejecutar todas las celdas en orden
3. Subir `images.zip` cuando se pida
4. Descargar los ficheros `*_raw.txt` al final

**En tu máquina local (después del notebook):**
```bash
# Copiar los ficheros descargados a data/ocr/
mv ~/Downloads/*_raw.txt data/ocr/

# FASE 4: estructurar a Markdown
uv run python -m refmate.ingest.structurer
```

In [ ]:
# ─── Celda 1: Dependencias ───────────────────────────────────────────────────
!pip install -q transformers accelerate torch pillow tqdm

In [ ]:
# ─── Celda 2: Verificar GPU ───────────────────────────────────────────────────
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No se detecta GPU. Ve a Runtime → Change runtime type → T4 GPU"
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ─── Celda 3: Subir imágenes ──────────────────────────────────────────────────
#
# OPCIÓN A (recomendada): subir images.zip
# OPCIÓN B: montar Google Drive (descomentar la sección correspondiente)
#
# ── Opción A: subir ZIP ──────────────────────────────────────────────────────
import os
from google.colab import files

print("Sube el fichero images.zip generado por FASE 2:")
uploaded = files.upload()

zip_name = next(iter(uploaded))
print(f"Descomprimiendo {zip_name}...")
!unzip -q "{zip_name}" -d .
print("Listo.")

# ── Opción B: Google Drive ───────────────────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# Ajusta esta ruta a donde tengas la carpeta images/ en tu Drive:
# !cp -r '/content/drive/MyDrive/refmate/images' .

In [ ]:
# ─── Celda 4: Verificar estructura de imágenes ────────────────────────────────
from pathlib import Path

IMAGES_DIR = Path("images")
DOC_IDS = ["reglas-de-juego", "rgc-fabm", "add-fabm"]

for doc_id in DOC_IDS:
    doc_dir = IMAGES_DIR / doc_id
    if not doc_dir.exists():
        print(f"  ✗ {doc_id}: directorio no encontrado en {doc_dir}")
        continue
    images = sorted(doc_dir.glob("page_*_crop.png"))
    print(f"  ✓ {doc_id}: {len(images)} imágenes")

In [ ]:
# ─── Celda 5: Cargar modelo ───────────────────────────────────────────────────
from transformers import LightOnOcrForConditionalGeneration, LightOnOcrProcessor

MODEL_ID = "lightonai/LightOnOCR-1B-1025"
DEVICE = "cuda"
DTYPE = torch.bfloat16
MAX_NEW_TOKENS = 4096

print(f"Cargando modelo {MODEL_ID}...")
processor = LightOnOcrProcessor.from_pretrained(MODEL_ID)
model = LightOnOcrForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
).to(DEVICE)
model.eval()
print("Modelo cargado.")

In [ ]:
# ─── Celda 6: Función OCR ─────────────────────────────────────────────────────
import time

def ocr_image(image_path: Path, retries: int = 3) -> str:
    """Extrae texto de una imagen con reintentos."""
    last_exc = None
    for attempt in range(1, retries + 1):
        try:
            conversation = [
                {"role": "user", "content": [{"type": "image", "url": str(image_path.resolve())}]}
            ]
            inputs = processor.apply_chat_template(
                conversation,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
            )
            inputs = {
                k: v.to(device=DEVICE, dtype=DTYPE) if v.is_floating_point() else v.to(DEVICE)
                for k, v in inputs.items()
            }
            with torch.no_grad():
                output_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)
            generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
            return processor.decode(generated_ids, skip_special_tokens=True).strip()
        except Exception as exc:
            last_exc = exc
            if attempt < retries:
                wait = 2 ** (attempt - 1)
                print(f"    Intento {attempt}/{retries} fallido: {exc}. Reintentando en {wait}s...")
                time.sleep(wait)
    raise RuntimeError(f"OCR fallido tras {retries} intentos: {last_exc}")

In [ ]:
# ─── Celda 7: Procesar documentos ────────────────────────────────────────────
#
# Produce data/ocr/{doc_id}_raw.txt con el formato exacto que espera
# el structurer (FASE 4): páginas separadas por \n\n<!-- PAGE N -->\n\n
#
from tqdm import tqdm

OUTPUT_DIR = Path("ocr_output")
OUTPUT_DIR.mkdir(exist_ok=True)

PAGE_MARKER = "\n\n<!-- PAGE {n} -->\n\n"

for doc_id in DOC_IDS:
    out_path = OUTPUT_DIR / f"{doc_id}_raw.txt"

    # Resume: si ya existe el fichero, saltar
    if out_path.exists():
        print(f"[{doc_id}] Ya procesado ({out_path}), saltando.")
        continue

    doc_dir = IMAGES_DIR / doc_id
    if not doc_dir.exists():
        print(f"[{doc_id}] Directorio no encontrado, saltando.")
        continue

    image_paths = sorted(doc_dir.glob("page_*_crop.png"))
    if not image_paths:
        print(f"[{doc_id}] Sin imágenes, saltando.")
        continue

    print(f"\n[{doc_id}] Procesando {len(image_paths)} páginas...")
    page_texts = []

    for image_path in tqdm(image_paths, desc=doc_id):
        try:
            text = ocr_image(image_path)
            page_texts.append(text)
        except Exception as exc:
            print(f"  ✗ {image_path.name}: {exc}")
            page_texts.append("")  # página vacía para mantener numeración

    # Ensamblar con marcadores de página (mismo formato que ocr_runner.py)
    parts = []
    for page_number, text in enumerate(page_texts, start=1):
        if page_number > 1:
            parts.append(PAGE_MARKER.format(n=page_number))
        parts.append(text)

    full_text = "".join(parts)
    out_path.write_text(full_text, encoding="utf-8")
    print(f"[{doc_id}] ✓ Guardado → {out_path} ({len(full_text):,} chars, {len(page_texts)} páginas)")

print("\nProcesamiento completado.")

In [ ]:
# ─── Celda 8: Verificar resultados ───────────────────────────────────────────
for doc_id in DOC_IDS:
    out_path = OUTPUT_DIR / f"{doc_id}_raw.txt"
    if out_path.exists():
        content = out_path.read_text(encoding="utf-8")
        page_count = content.count("<!-- PAGE ")
        print(f"  ✓ {doc_id}_raw.txt: {len(content):,} chars, {page_count + 1} páginas")
    else:
        print(f"  ✗ {doc_id}_raw.txt: NO GENERADO")

In [ ]:
# ─── Celda 9: Descargar resultados ───────────────────────────────────────────
for doc_id in DOC_IDS:
    out_path = OUTPUT_DIR / f"{doc_id}_raw.txt"
    if out_path.exists():
        files.download(str(out_path))
        print(f"Descargando {out_path.name}...")

In [ ]:
# ─── Celda 10: Preview (opcional) ────────────────────────────────────────────
# Ver las primeras líneas de cada fichero para verificar calidad del OCR

for doc_id in DOC_IDS:
    out_path = OUTPUT_DIR / f"{doc_id}_raw.txt"
    if out_path.exists():
        content = out_path.read_text(encoding="utf-8")
        print(f"\n{'='*60}")
        print(f"  {doc_id}")
        print('='*60)
        print(content[:1000])
        print("...")